In [ ]:
# 05_export_viewer.ipynb

# ---- Cell 1: Install and initialize ----
!pip install -q earthengine-api geopandas shapely

import ee, time, os, json, zipfile, datetime
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import shape
from google.colab import files

ee.Authenticate()
PROJECT_ID = 'your-cloud-project-id'
ee.Initialize(project=PROJECT_ID)
print('Earth Engine ready')

# ---- Cell 2: Settings ----
UJANI_LAT, UJANI_LON = 18.0, 75.1
centre = ee.Geometry.Point([UJANI_LON, UJANI_LAT])
aoi = centre.buffer(10000).bounds()

EXPORT_DIR = 'web_export'
DATA_DIR = os.path.join(EXPORT_DIR, 'data')
GEO_DIR = os.path.join(EXPORT_DIR, 'geo')
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(GEO_DIR, exist_ok=True)

NDWI_THRESHOLD = 0.2      # must match notebook 02
N_POLYGON_MONTHS = 6      # keep small; each month is one server request
SIMPLIFY_DEG = 0.0002     # about 20 m; reduces file size
UTM_EPSG = 32643          # UTM zone 43N, used only for area calculations

# ---- Cell 3: Export tabular data for the viewer ----
def clean_and_save(src, dst, keep, round_to=2):
    df = pd.read_csv(src)
    cols = [c for c in keep if c in df.columns]
    df = df[cols].round(round_to)
    df.to_csv(os.path.join(DATA_DIR, dst), index=False)
    print(f'{dst}: {len(df)} rows, columns {cols}')
    return df

water = pd.read_csv('ujani_water_extent_monthly.csv')
water['water_km2'] = water['s2_water_km2'].fillna(water['s1_water_km2'])
water['source'] = np.where(water['s2_water_km2'].notna(), 'Sentinel-2',
                   np.where(water['s1_water_km2'].notna(), 'Sentinel-1', ''))
water = water[['month', 'water_km2', 'source', 's2_water_km2', 's1_water_km2']].round(3)
water.to_csv(os.path.join(DATA_DIR, 'water_extent_monthly.csv'), index=False)
print(f'water_extent_monthly.csv: {len(water)} rows')

clean_and_save('ujani_forecast_6month.csv', 'forecast_6month.csv',
               ['date', 'water_km2_forecast'])

clean_and_save('ujani_twin_baseline.csv', 'twin_baseline.csv',
               ['date', 'rain_mm', 'inflow_mcm', 'release_mcm',
                'evap_mcm', 'volume_mcm', 'percent_full'])

clean_and_save('ujani_model_dataset.csv', 'model_dataset.csv',
               ['month', 'date', 'water_km2', 'rain_mm', 'source'])

# ---- Cell 4: Export the study area as GeoJSON ----
aoi_geom = aoi.getInfo()
study_area = {
    'type': 'Feature',
    'properties': {'name': 'Ujani study area (10 km radius, bounding box)',
                   'centre_lat': UJANI_LAT, 'centre_lon': UJANI_LON},
    'geometry': aoi_geom
}
with open(os.path.join(GEO_DIR, 'study_area.geojson'), 'w') as f:
    json.dump(study_area, f)
print('study_area.geojson saved')

# ---- Cell 5: Water-extent polygons for selected months (one request per month) ----
s2_col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
          .filterBounds(aoi)
          .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 40)))

def mask_s2(img):
    scl = img.select('SCL')
    clear = (scl.neq(0).And(scl.neq(3)).And(scl.neq(8))
                 .And(scl.neq(9)).And(scl.neq(10)))
    return img.updateMask(clear).copyProperties(img, ['system:time_start'])

def water_image(month_str):
    start = ee.Date(month_str)
    comp = s2_col.filterDate(start, start.advance(1, 'month')).map(mask_s2).median()
    ndwi = comp.normalizedDifference(['B3', 'B8'])
    return ndwi.gt(NDWI_THRESHOLD).selfMask().clip(aoi)

def month_polygons(month_str):
    vec = water_image(month_str).reduceToVectors(
        geometry=aoi,
        scale=20,
        geometryType='polygon',
        eightConnected=False,
        maxPixels=1e9,
        bestEffort=True)
    return vec.getInfo()

def call_with_retry(fn, arg, retries=5):
    for attempt in range(retries):
        try:
            return fn(arg)
        except Exception as e:
            msg = str(e).lower()
            if any(k in msg for k in ['concurrent', 'quota', 'rate', 'too many']) and attempt < retries - 1:
                wait = 10 * (2 ** attempt)
                print(f'  limit reached, waiting {wait}s...')
                time.sleep(wait)
            else:
                raise

# Pick evenly spaced months that have Sentinel-2 data
available = water.dropna(subset=['s2_water_km2'])['month'].tolist()
idx = np.linspace(0, len(available) - 1, min(N_POLYGON_MONTHS, len(available))).round().astype(int)
selected = sorted(set(available[i] for i in idx))
print('Months exported as polygons:', selected)

polygon_results = {}
for m in selected:
    polygon_results[m] = call_with_retry(month_polygons, f'{m}-01')
    print(f'{m}: {len(polygon_results[m].get("features", []))} polygon(s)')
    time.sleep(1)

# ---- Cell 6: Simplify, add area, and write GeoJSON ----
rows = []
for m, fc in polygon_results.items():
    for feat in fc.get('features', []):
        geom = shape(feat['geometry'])
        if not geom.is_empty:
            rows.append({'month': m, 'geometry': geom})

if rows:
    gdf = gpd.GeoDataFrame(rows, crs='EPSG:4326')
    gdf['geometry'] = gdf.geometry.simplify(SIMPLIFY_DEG, preserve_topology=True)
    gdf = gdf.dissolve(by='month').reset_index()
    gdf['area_km2'] = (gdf.to_crs(UTM_EPSG).area / 1e6).round(3)
    out_path = os.path.join(GEO_DIR, 'water_extent_polygons.geojson')
    gdf.to_file(out_path, driver='GeoJSON')
    print(gdf[['month', 'area_km2']])
    print(f'Saved {out_path}')
else:
    print('No water polygons found. Check the NDWI threshold and the selected months.')

# ---- Cell 7: Metadata (sources, methods, assumptions, limits) ----
meta = {
    'project': 'TerraVita AquaTwin demo: Ujani Reservoir',
    'generated_utc': datetime.datetime.utcnow().isoformat(timespec='seconds') + 'Z',
    'study_area': {'centre_lat': UJANI_LAT, 'centre_lon': UJANI_LON,
                   'radius_km': 10, 'crs': 'EPSG:4326'},
    'sources': {
        'optical_water': 'Sentinel-2 SR Harmonized (Copernicus), NDWI',
        'radar_water': 'Sentinel-1 GRD VV (Copernicus)',
        'rainfall': 'CHIRPS daily, summed to months',
        'study_area_location': 'Approximate; replace with exact boundary when available'
    },
    'methods': {
        'ndwi_threshold': NDWI_THRESHOLD,
        'vv_threshold_db': -18,
        'cloud_mask': 'Sentinel-2 Scene Classification Layer',
        'monthly_composite': 'Median of cloud-masked images'
    },
    'assumptions': {
        'mean_depth_m': 10.0,
        'full_capacity_mcm': 117.0,
        'catchment_km2': 1200.0,
        'runoff_coefficient': 0.15,
        'evap_mm_per_month': 150.0,
        'note': 'Placeholder values. Replace with published Ujani figures and cite them.'
    },
    'limitations': [
        'Water masks depend on the chosen threshold and on cloud or wind conditions.',
        'Storage volumes are estimated from surface area and an assumed average depth.',
        'The twin is a simplified water balance, not a dam engineering model.',
        'The forecast uses average rainfall for future months and is not a prediction of actual conditions.',
        'Polygons cover the rectangular study area only, so shapes are cut at its edges.'
    ]
}
with open(os.path.join(EXPORT_DIR, 'metadata.json'), 'w') as f:
    json.dump(meta, f, indent=2)
print('metadata.json saved')

# ---- Cell 8: Validate the export ----
for root, _, fs in os.walk(EXPORT_DIR):
    for f in sorted(fs):
        p = os.path.join(root, f)
        print(f'{p:50s} {os.path.getsize(p) / 1024:8.1f} KB')

check = pd.read_csv(os.path.join(DATA_DIR, 'water_extent_monthly.csv'))
print(f"\nWater CSV rows: {len(check)}, missing S2 values: {check['s2_water_km2'].isna().sum()}")

# ---- Cell 9: Zip and download ----
zip_path = 'terravita_ujani_web_export.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, fs in os.walk(EXPORT_DIR):
        for f in fs:
            p = os.path.join(root, f)
            z.write(p, p)
print(f'Created {zip_path}')
files.download(zip_path)